# Final untouched HOLDOUT evaluation

This is the first terminal assessment of the fully frozen TRAIN-only 467-feature ensemble. The committed system lock is checked before HOLDOUT access. The primary action is Balanced expected-cost decisioning with rules disabled. The optional frozen-rule overlay is reported separately. No tuning is permitted.

In [1]:
import gc
import hashlib
import json
import sys
import time
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from fraud_detection.data import merge_train_data
from fraud_detection.decisioning.costs import scenario_profiles
from fraud_detection.decisioning.policy import decide
from fraud_detection.decisioning.rules import Rule, RulesEngine, RuleThresholds
from fraud_detection.evaluation.holdout import prediction_artifact
from fraud_detection.features import COMPOSITE_TARGET_ENTITIES
from fraud_detection.features.composite import CompositeIdentity, create_composite_keys
from fraud_detection.models import prepare_engineered_matrices
from fraud_detection.models.ensemble import weighted_score_blend

started = time.perf_counter()
full_rows = 590540
train_end = int(full_rows * 0.70)
validation_end = train_end + int(full_rows * 0.15)
expected_holdout_rows = full_rows - validation_end
assert expected_holdout_rows == 88581
prediction_path = ROOT / "reports/holdout_predictions_frozen.csv"
lock_path = ROOT / "reports/holdout_prediction_lock.json"
if prediction_path.exists() or lock_path.exists():
    raise RuntimeError("Terminal prediction artifact already exists; do not rerun inference")
manifest = (ROOT / "reports/frozen_system_manifest.md").read_text(encoding="utf-8")
assert "LightGBM: rebuild and fit on **TRAIN only**" in manifest
assert "XGBoost: rebuild and fit on **TRAIN only**" in manifest
assert "**DISABLED BY DEFAULT**" in manifest
frozen_config_bytes = (ROOT / "reports/rules_frozen_config.json").read_bytes()
assert hashlib.sha256(frozen_config_bytes).hexdigest() == (
    "3a4380831e177828c82a1a118259f83e354221fa52205b41d38aa21b58febb94"
)
calibrator_path = ROOT / "models/probability_calibrator.joblib"
assert hashlib.sha256(calibrator_path.read_bytes()).hexdigest() == (
    "3503bf66161b5152e9de4d638b9e7932d581fc371abd6a45b6969ba6c8134f6e"
)
calibrator = joblib.load(calibrator_path)
assert calibrator.method == "platt"
profiles = scenario_profiles()
transaction_path = ROOT / "data/raw/train_transaction.csv"
sample = pd.read_csv(transaction_path, nrows=1000, low_memory=False)
dtypes = {
    name: ("float64" if pd.api.types.is_numeric_dtype(dtype) else "object")
    for name, dtype in sample.dtypes.items()
}
dtypes["TransactionID"] = "int64"
dtypes["TransactionDT"] = "int64"
dtypes["card1"] = "int64"
dtypes["isFraud"] = "int8"
del sample
train_transaction = pd.read_csv(transaction_path, nrows=train_end, dtype=dtypes)
holdout_transaction = pd.read_csv(
    transaction_path,
    skiprows=range(1, validation_end + 1),
    nrows=expected_holdout_rows,
    usecols=lambda column: column != "isFraud",
    dtype={name: dtype for name, dtype in dtypes.items() if name != "isFraud"},
)
development_boundary = pd.read_csv(
    transaction_path,
    nrows=validation_end,
    usecols=["TransactionID", "TransactionDT", "TransactionAmt"],
)
assert len(train_transaction) == train_end
assert len(holdout_transaction) == expected_holdout_rows
assert train_transaction.TransactionID.is_unique
assert holdout_transaction.TransactionID.is_unique
assert not holdout_transaction.TransactionID.isin(development_boundary.TransactionID).any()
assert holdout_transaction.TransactionID.min() > development_boundary.TransactionID.max()
assert holdout_transaction.TransactionDT.min() >= development_boundary.TransactionDT.max()
assert set(train_transaction.columns) - {"isFraud"} == set(holdout_transaction.columns)
assert "isFraud" not in holdout_transaction
validation_amount = development_boundary.TransactionAmt.iloc[train_end:].copy()
del development_boundary
identity_path = ROOT / "data/raw/train_identity.csv"
identity = pd.read_csv(identity_path, low_memory=False)
assert identity.TransactionID.is_unique
combined_transaction = pd.concat(
    [train_transaction, holdout_transaction], ignore_index=True, copy=False
)
assert combined_transaction.isFraud.iloc[train_end:].isna().all()
identity = identity.loc[identity.TransactionID.isin(combined_transaction.TransactionID)]
merged = merge_train_data(combined_transaction, identity)
train = merged.iloc[:train_end]
holdout_features = merged.iloc[train_end:].drop(columns="isFraud")
assert set(train.columns) - {"isFraud"} == set(holdout_features.columns)
assert "isFraud" not in holdout_features
del identity, combined_transaction, merged, train_transaction, holdout_transaction
gc.collect()
amount_edges = np.r_[
    0.0,
    np.unique(train.TransactionAmt.quantile([0.25, 0.50, 0.75, 0.90]).to_numpy()),
    np.inf,
]

print("Stage 1 complete: locked inputs and chronology verified")

Stage 1 complete: locked inputs and chronology verified


In [2]:
prepared = prepare_engineered_matrices(
    train,
    holdout_features,
    include_graph=False,
    include_composite=True,
    include_target_encoding=True,
    target_encoding_label_delay_days=0.0,
    scaling_batch_rows=len(train),
)
positions = {name: i for i, name in enumerate(prepared.feature_names)}
baseline_names = tuple(
    name for name in prepared.feature_names if not name.endswith("_target_encoded_fraud_rate")
)
composite_te_features = tuple(
    f"{entity}_target_encoded_fraud_rate" for entity in COMPOSITE_TARGET_ENTITIES
)
selected_names = baseline_names + composite_te_features
assert len(selected_names) == 467 and len(set(selected_names)) == 467
assert "isFraud" not in selected_names and "y_true" not in selected_names
indices = np.asarray([positions[name] for name in selected_names], dtype=np.int32)
x_train, x_holdout = prepared.x_train[:, indices], prepared.x_evaluation[:, indices]
y_train = train.isFraud.astype("int8")
matrix_mib = (prepared.x_train.nbytes + prepared.x_evaluation.nbytes) / 1024**2

print("Stage 2 complete: frozen 467-feature matrices prepared")

Stage 2 complete: frozen 467-feature matrices prepared


In [3]:
lgbm = lgb.LGBMClassifier(
    learning_rate=0.1,
    n_estimators=400,
    num_leaves=63,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)
lgbm.fit(x_train, y_train)
lgbm_score = lgbm.predict_proba(x_holdout)[:, 1]
scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
assert abs(scale_pos_weight - 27.434310084) < 1e-8
xgboost = xgb.XGBClassifier(
    learning_rate=0.1,
    max_depth=7,
    n_estimators=300,
    tree_method="hist",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
)
xgboost.fit(x_train, y_train)
xgboost_score = xgboost.predict_proba(x_holdout)[:, 1]
ensemble_score = weighted_score_blend(
    lgbm_score, xgboost_score, lightgbm_weight=0.8, xgboost_weight=0.2
)
calibrated_probability = calibrator.transform(ensemble_score)

print("Stage 3 complete: TRAIN-only component predictions and calibration")

Stage 3 complete: TRAIN-only component predictions and calibration


In [4]:
decision_input = pd.DataFrame(
    {
        "TransactionID": holdout_features.TransactionID.to_numpy(),
        "calibrated_probability": calibrated_probability,
        "TransactionAmt": holdout_features.TransactionAmt.to_numpy(),
    }
)
base = decide(decision_input, profiles["Balanced"])
base_capped = decide(decision_input, profiles["Balanced"], maximum_review_rate=0.05)
artifact = prediction_artifact(
    holdout_features, lgbm_score, xgboost_score, ensemble_score, calibrated_probability, base
)
config = json.loads(frozen_config_bytes)
engine = RulesEngine(
    RuleThresholds(**config["thresholds"]), tuple(Rule(**rule) for rule in config["rules"])
)
composite = (CompositeIdentity("card1_addr1_p_emaildomain", ("card1", "addr1", "P_emaildomain")),)
train_keys = create_composite_keys(train, composite).iloc[:, 0]
holdout_keys = create_composite_keys(holdout_features, composite).iloc[:, 0]
reference_frequency = train_keys.value_counts(normalize=True)
rule_signals = artifact.loc[
    :,
    [
        "TransactionID",
        "calibrated_probability",
        "TransactionAmt",
        "lightgbm_score",
        "xgboost_score",
    ],
].copy()
rule_signals["composite_frequency"] = holdout_keys.map(reference_frequency).fillna(0).to_numpy()
rule_signals["base_decision"] = base.decision.to_numpy()
optional_rules = engine.evaluate(rule_signals)
assert np.array_equal(optional_rules.calibrated_probability, artifact.calibrated_probability)
artifact["base_5pct_decision"] = base_capped.decision.to_numpy(copy=True)
artifact["optional_rules_decision"] = optional_rules.final_decision.to_numpy(copy=True)
artifact["optional_triggered_rule_ids"] = optional_rules.triggered_rule_ids.map(
    lambda ids: "|".join(ids)
).to_numpy()
assert "isFraud" not in artifact
artifact.to_csv(prediction_path, index=False)
prediction_hash = hashlib.sha256(prediction_path.read_bytes()).hexdigest()
lock = {
    "prediction_file": prediction_path.name,
    "sha256": prediction_hash,
    "row_count": len(artifact),
    "label_column_present": False,
    "model_fit_scope": "TRAIN only",
    "primary_policy": "Balanced, rules disabled",
    "optional_rules": "frozen overlay only",
    "amount_band_edges_train": [float(x) if np.isfinite(x) else "infinity" for x in amount_edges],
}
lock_path.write_text(json.dumps(lock, indent=2) + "\n", encoding="utf-8")
PREDICTIONS_LOCKED = True
print("Prediction artifact locked BEFORE label access:", prediction_path.name)
print("SHA-256:", prediction_hash, "rows:", len(artifact))
print("Prepared feature matrix MiB:", round(matrix_mib, 1))
display(artifact.head(3).drop(columns="TransactionID"))

Prediction artifact locked BEFORE label access: holdout_predictions_frozen.csv
SHA-256: 0d7f5d117436cd7227d682388c5cceddbc72b689143984a7d186d9ec03c1391f rows: 88581
Prepared feature matrix MiB: 905.7


,TransactionDT,TransactionAmt,lightgbm_score,xgboost_score,ensemble_score,calibrated_probability,base_decision,base_5pct_decision,optional_rules_decision,optional_triggered_rule_ids
0,13151880,57.95,0.000188,0.001534,0.000457,0.000558,APPROVE,APPROVE,APPROVE,
1,13151945,47.95,0.027412,0.107768,0.043484,0.020108,APPROVE,APPROVE,APPROVE,
2,13152098,209.95,0.109161,0.195661,0.126461,0.048402,REVIEW,APPROVE,REVIEW,


## Prediction lock and label reveal

The preceding cell saves the label-free prediction artifact and SHA-256 lock. Only now may HOLDOUT labels be read for terminal metrics. The frozen primary policy is never revised based on those outcomes.

In [5]:
import hashlib
import json
import time

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score

from fraud_detection.decisioning.evaluation import evaluate_policy
from fraud_detection.evaluation.holdout import diagnostic_cutoff_metrics
from fraud_detection.models.calibration import evaluate as evaluate_calibration
from fraud_detection.models.calibration import reliability_bins

assert PREDICTIONS_LOCKED and prediction_path.exists() and lock_path.exists()
locked = json.loads(lock_path.read_text(encoding="utf-8"))
assert locked["sha256"] == hashlib.sha256(prediction_path.read_bytes()).hexdigest()
assert locked["label_column_present"] is False
assert not any(column == "isFraud" for column in pd.read_csv(prediction_path, nrows=0).columns)
# The first access to HOLDOUT labels occurs only after the saved prediction hash is verified.
holdout_labels = pd.read_csv(
    transaction_path,
    skiprows=range(1, validation_end + 1),
    nrows=expected_holdout_rows,
    usecols=["TransactionID", "isFraud"],
)
assert len(holdout_labels) == len(artifact)
assert holdout_labels.TransactionID.is_unique
assert np.array_equal(holdout_labels.TransactionID, artifact.TransactionID)
y_holdout = holdout_labels.isFraud.to_numpy(dtype=np.int8)
assert np.isin(y_holdout, (0, 1)).all()
ranking = {
    "pr_auc": float(average_precision_score(y_holdout, artifact.ensemble_score)),
    "roc_auc": float(roc_auc_score(y_holdout, artifact.ensemble_score)),
}
calibration = evaluate_calibration(y_holdout, artifact.calibrated_probability)
diagnostic = diagnostic_cutoff_metrics(y_holdout, artifact.calibrated_probability)
balanced = profiles["Balanced"]
primary = evaluate_policy(base, y_holdout, balanced)
capacity = evaluate_policy(base_capped, y_holdout, balanced)


def policy_view(base_frame, decisions):
    view = base_frame.copy(deep=True)
    action = np.asarray(decisions)
    approve = view.expected_cost_approve.to_numpy()
    review = view.expected_cost_review.to_numpy()
    block = view.expected_cost_block.to_numpy()
    view["decision"] = action
    view["selected_expected_cost"] = np.where(
        action == "APPROVE", approve, np.where(action == "REVIEW", review, block)
    )
    return view


optional_view = policy_view(base, optional_rules.final_decision)
optional = evaluate_policy(optional_view, y_holdout, balanced)
assert np.array_equal(artifact.base_decision, base.decision)
assert np.array_equal(artifact.optional_rules_decision, optional_rules.final_decision)
rule_trigger_count = int(optional_rules.triggered_rule_ids.map(bool).sum())
rule_override_count = int(optional_rules.rule_override.sum())
rule_table = pd.DataFrame(
    [
        {
            "rule_id": rule.rule_id,
            "trigger_count": int(
                optional_rules.triggered_rule_ids.map(
                    lambda ids, rule_id=rule.rule_id: rule_id in ids
                ).sum()
            ),
        }
        for rule in engine.rules
    ]
)
rule_table["trigger_rate"] = rule_table.trigger_count / len(artifact)
rank_table = pd.DataFrame(
    [
        {
            "n": len(artifact),
            "pr_auc": ranking["pr_auc"],
            "roc_auc": ranking["roc_auc"],
            "validation_pr_auc": 0.615265,
            "validation_roc_auc": 0.916224,
            "pr_auc_absolute_change": ranking["pr_auc"] - 0.615265,
            "pr_auc_relative_change": ranking["pr_auc"] / 0.615265 - 1,
            "roc_auc_absolute_change": ranking["roc_auc"] - 0.916224,
            "roc_auc_relative_change": ranking["roc_auc"] / 0.916224 - 1,
        }
    ]
)
calibration_table = pd.DataFrame(
    [
        {
            key: calibration[key]
            for key in (
                "brier",
                "log_loss",
                "ece",
                "mean_probability",
                "prevalence",
                "calibration_gap",
            )
        }
    ]
)
policy_table = pd.DataFrame(
    [
        {"policy": "primary_balanced_rules_disabled", **primary},
        {"policy": "primary_5pct_review_capacity", **capacity},
        {"policy": "optional_frozen_rules_overlay", **optional},
    ]
)
rank_table.to_csv(ROOT / "reports/holdout_ranking_metrics.csv", index=False)
calibration_table.to_csv(ROOT / "reports/holdout_calibration_metrics.csv", index=False)
policy_table.iloc[:2].to_csv(ROOT / "reports/holdout_business_policy_metrics.csv", index=False)
policy_table.iloc[2:].to_csv(ROOT / "reports/holdout_optional_rules_metrics.csv", index=False)

validation_reference = {
    "split": "VALIDATION",
    "fraud_prevalence": 0.034341,
    "pr_auc": 0.615265,
    "roc_auc": 0.916224,
    "brier": 0.019493,
    "log_loss": 0.082039,
    "ece": 0.005434,
    "mean_calibrated_probability": 0.036660,
    "approve_rate": 0.834355,
    "review_rate": 0.147131,
    "block_rate": 0.018514,
    "fraud_capture_rate": 0.690138,
    "fraud_amount_capture_rate": 0.817614,
    "legitimate_review_rate": 0.138709,
    "legitimate_block_rate": 0.005553,
    "modeled_cost_per_transaction": 2.025542,
}
holdout_comparison = {
    "split": "HOLDOUT",
    "fraud_prevalence": calibration["prevalence"],
    "pr_auc": ranking["pr_auc"],
    "roc_auc": ranking["roc_auc"],
    "brier": calibration["brier"],
    "log_loss": calibration["log_loss"],
    "ece": calibration["ece"],
    "mean_calibrated_probability": calibration["mean_probability"],
    **{
        key: primary[key]
        for key in (
            "approve_rate",
            "review_rate",
            "block_rate",
            "fraud_capture_rate",
            "fraud_amount_capture_rate",
            "legitimate_review_rate",
            "legitimate_block_rate",
            "modeled_cost_per_transaction",
        )
    },
}
comparison = pd.DataFrame([validation_reference, holdout_comparison])
comparison.to_csv(ROOT / "reports/holdout_validation_comparison.csv", index=False)
amount_band = pd.cut(
    artifact.TransactionAmt,
    bins=amount_edges,
    include_lowest=True,
    right=False,
    labels=[
        f"[{a:.2f}, {b:.2f})" if np.isfinite(b) else f"[{a:.2f}, +inf)"
        for a, b in zip(amount_edges[:-1], amount_edges[1:], strict=True)
    ],
)
band_rows = []
for band in amount_band.cat.categories:
    selected = (amount_band == band).to_numpy()
    fraud = selected & (y_holdout == 1)
    amount = artifact.TransactionAmt.to_numpy(dtype=float)
    actions = base.decision.to_numpy()
    fraud_amount = float(amount[fraud].sum())
    captured_count = float(
        (fraud & (actions == "BLOCK")).sum()
        + balanced.review_fraud_detection_rate * (fraud & (actions == "REVIEW")).sum()
    )
    captured_amount = float(
        amount[fraud & (actions == "BLOCK")].sum()
        + balanced.review_fraud_detection_rate * amount[fraud & (actions == "REVIEW")].sum()
    )
    band_rows.append(
        {
            "amount_band": band,
            "transactions": int(selected.sum()),
            "fraud_count": int(fraud.sum()),
            "fraud_amount": fraud_amount,
            "modeled_fraud_transaction_capture": captured_count / fraud.sum() if fraud.any() else 0,
            "modeled_fraud_amount_capture": captured_amount / fraud_amount if fraud_amount else 0,
            "approved_fraud_amount": float(amount[fraud & (actions == "APPROVE")].sum()),
        }
    )
band_table = pd.DataFrame(band_rows)
band_table.to_csv(ROOT / "reports/holdout_amount_band_outcomes.csv", index=False)

bins = reliability_bins(y_holdout, artifact.calibrated_probability)
fig_dir = ROOT / "reports/figures"
fig_dir.mkdir(parents=True, exist_ok=True)
precision, recall, _ = precision_recall_curve(y_holdout, artifact.ensemble_score)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall, precision, color="#3176ab")
ax.axhline(y_holdout.mean(), color="gray", ls="--", label="Prevalence")
ax.set(xlabel="Recall", ylabel="Precision", title="Terminal HOLDOUT precision-recall curve")
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir / "holdout_precision_recall_curve.png", dpi=170)
plt.close(fig)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, 1], [0, 1], ls="--", color="gray", label="Perfect calibration")
ax.scatter(
    bins.mean_probability,
    bins.fraud_rate,
    s=60,
    color="#3176ab",
    label="Frozen Platt probabilities",
)
ax.set(
    xlabel="Mean calibrated probability",
    ylabel="Observed fraud rate",
    title="Terminal HOLDOUT reliability (10 equal-count bins)",
)
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir / "holdout_calibration_curve.png", dpi=170)
plt.close(fig)
fig, ax = plt.subplots(figsize=(7, 5))
locations = np.arange(2)
for i, metric in enumerate(("pr_auc", "roc_auc")):
    ax.bar(locations + (i - 0.5) * 0.35, comparison[metric], 0.35, label=metric.upper())
ax.set_xticks(locations, comparison.split)
ax.set(ylabel="Score", title="Frozen VALIDATION versus terminal HOLDOUT ranking")
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir / "validation_vs_holdout_metrics.png", dpi=170)
plt.close(fig)
fig, ax = plt.subplots(figsize=(7, 5))
bottom = np.zeros(2)
for name, color in (("approve", "#7ba58c"), ("review", "#e7aa57"), ("block", "#ba6870")):
    values = [validation_reference[f"{name}_rate"], primary[f"{name}_rate"]]
    ax.bar(["VALIDATION", "HOLDOUT"], values, bottom=bottom, label=name.upper(), color=color)
    bottom += values
ax.set(ylabel="Share of transactions", title="Primary Balanced actions; rules disabled")
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir / "holdout_decision_distribution.png", dpi=170)
plt.close(fig)
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(band_table.amount_band.astype(str), band_table.modeled_fraud_amount_capture, color="#3176ab")
ax.set(
    xlabel="TRAIN-defined amount band",
    ylabel="Modeled fraud amount capture",
    title="Primary policy HOLDOUT fraud amount capture by amount",
)
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(fig_dir / "holdout_fraud_amount_capture.png", dpi=170)
plt.close(fig)

try:
    import psutil

    kernel_rss_mib = psutil.Process().memory_info().rss / 1024**2
except ImportError:
    kernel_rss_mib = float("nan")
runtime_seconds = time.perf_counter() - started
tracking_uri = f"sqlite:///{(ROOT / 'model_comparison.db').resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("fraud-detection-final-holdout")
with mlflow.start_run(run_name="terminal-train-only-balanced-rules-disabled") as run:
    mlflow.set_tags(
        {
            "stage": "final-holdout",
            "dataset": "ieee-cis",
            "evaluation": "untouched-chronological-holdout",
            "model": "frozen-calibrated-ensemble",
            "primary_policy": "balanced-rules-disabled",
        }
    )
    mlflow.log_params(
        {
            "fit_scope": "TRAIN only",
            "feature_count": 467,
            "lightgbm_weight": 0.8,
            "xgboost_weight": 0.2,
            "calibration": "frozen-platt",
            "rules_primary": "disabled",
            "prediction_sha256": prediction_hash,
        }
    )
    mlflow.log_metrics(
        {
            "holdout_rows": len(artifact),
            **ranking,
            **{
                key: calibration[key]
                for key in ("brier", "log_loss", "ece", "mean_probability", "prevalence")
            },
            **{f"diagnostic_0_50_{key}": value for key, value in diagnostic.items()},
            **{
                f"primary_{key}": primary[key]
                for key in (
                    "approve_rate",
                    "review_rate",
                    "block_rate",
                    "fraud_capture_rate",
                    "fraud_amount_capture_rate",
                    "legitimate_review_rate",
                    "legitimate_block_rate",
                    "modeled_total_cost",
                    "modeled_cost_per_transaction",
                )
            },
            "capacity_5pct_review_rate": capacity["review_rate"],
            "capacity_5pct_modeled_cost": capacity["modeled_total_cost"],
            "optional_rules_trigger_rate": rule_trigger_count / len(artifact),
            "optional_rules_override_rate": rule_override_count / len(artifact),
            "optional_rules_fraud_capture_rate": optional["fraud_capture_rate"],
            "optional_rules_modeled_cost": optional["modeled_total_cost"],
            "runtime_seconds": runtime_seconds,
        }
    )
    mlflow.log_artifact(str(lock_path))
    for name in (
        "holdout_precision_recall_curve",
        "holdout_calibration_curve",
        "validation_vs_holdout_metrics",
        "holdout_decision_distribution",
        "holdout_fraud_amount_capture",
    ):
        mlflow.log_artifact(str(fig_dir / f"{name}.png"), artifact_path="figures")
    run_id = run.info.run_id
print("Terminal HOLDOUT MLflow run:", run_id)
display(rank_table, calibration_table, policy_table, comparison, band_table)

2026/10/12 11:51:25 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-final-holdout' does not exist. Creating a new experiment.


Terminal HOLDOUT MLflow run: 447dbfb1edac4b8d9db70850b6c4b9bb


,n,pr_auc,roc_auc,validation_pr_auc,validation_roc_auc,pr_auc_absolute_change,pr_auc_relative_change,roc_auc_absolute_change,roc_auc_relative_change
0,88581,0.5369,0.878637,0.615265,0.916224,-0.078365,-0.127367,-0.037587,-0.041024


,brier,log_loss,ece,mean_probability,prevalence,calibration_gap
0,0.021823,0.094295,0.005559,0.037656,0.034804,0.002852


,policy,n,fraud_count,legitimate_count,fraud_amount_total,fraud_capture_rate,fraud_amount_capture_rate,fraud_transactions_approved,fraud_transactions_reviewed,fraud_transactions_blocked,...,review_overflow_count,approve_count,approve_rate,legitimate_approve_rate,review_count,review_rate,legitimate_review_rate,block_count,block_rate,legitimate_block_rate
0,primary_balanced_rules_disabled,88581,3083,85498,469608.521,0.622511,0.764860,945,1094,1044,...,0,73351,0.828067,0.846874,13595,0.153475,0.146214,1635,0.018458,0.006912
1,primary_5pct_review_capacity,88581,3083,85498,469608.521,0.526370,0.683765,1391,346,1346,...,9166,81470,0.919723,0.936618,4429,0.049999,0.047756,2682,0.030277,0.015626
2,optional_frozen_rules_overlay,88581,3083,85498,469608.521,0.664548,0.774638,783,1256,1044,...,0,70377,0.794493,0.813984,16569,0.187049,0.179104,1635,0.018458,0.006912


,split,fraud_prevalence,pr_auc,roc_auc,brier,log_loss,ece,mean_calibrated_probability,approve_rate,review_rate,block_rate,fraud_capture_rate,fraud_amount_capture_rate,legitimate_review_rate,legitimate_block_rate,modeled_cost_per_transaction
0,VALIDATION,0.034341,0.615265,0.916224,0.019493,0.082039,0.005434,0.036660,0.834355,0.147131,0.018514,0.690138,0.817614,0.138709,0.005553,2.025542
1,HOLDOUT,0.034804,0.536900,0.878637,0.021823,0.094295,0.005559,0.037656,0.828067,0.153475,0.018458,0.622511,0.764860,0.146214,0.006912,2.067025


,amount_band,transactions,fraud_count,fraud_amount,modeled_fraud_transaction_capture,modeled_fraud_amount_capture,approved_fraud_amount
0,"[0.00, 42.95)",21580,1034,24248.296,0.538104,0.571175,9014.097
1,"[42.95, 68.95)",22756,544,29447.046,0.536397,0.535757,11831.929
2,"[68.95, 125.00)",22521,658,64661.190,0.600608,0.590071,22132.896
3,"[125.00, 280.00)",13163,412,76661.923,0.755825,0.755391,12041.352
4,"[280.00, +inf)",8561,435,274590.066,0.837701,0.850336,9832.270


## Terminal report

The tables below summarize ranking, calibration, primary Balanced actions, the fixed 5% review-capacity sensitivity, the optional frozen operational overlay, and TRAIN-defined amount bands.

In [6]:
def markdown_table(table, columns):
    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join(["---"] * len(columns)) + " |",
    ]
    for row in table.loc[:, columns].itertuples(index=False, name=None):
        lines.append(
            "| "
            + " | ".join(
                f"{value:.6f}" if isinstance(value, float) else str(value) for value in row
            )
            + " |"
        )
    return "\n".join(lines)


comparison_columns = list(comparison.columns)
policy_columns = [
    "policy",
    "approve_count",
    "review_count",
    "block_count",
    "approve_rate",
    "review_rate",
    "block_rate",
    "fraud_capture_rate",
    "fraud_amount_capture_rate",
    "fraud_transactions_approved",
    "fraudulent_amount_approved",
    "legitimate_approve_rate",
    "legitimate_review_rate",
    "legitimate_block_rate",
    "modeled_total_cost",
    "modeled_cost_per_transaction",
    "approve_all_modeled_cost",
    "modeled_cost_reduction",
]
diagnostic_text = ", ".join(
    f"{key}={value:.6f}" if isinstance(value, float) else f"{key}={value}"
    for key, value in diagnostic.items()
)
train_amount_stats = train.TransactionAmt.quantile([0.5, 0.9, 0.99]).to_dict()
validation_amount_stats = validation_amount.quantile([0.5, 0.9, 0.99]).to_dict()
holdout_amount_stats = artifact.TransactionAmt.quantile([0.5, 0.9, 0.99]).to_dict()
amount_shift = pd.DataFrame(
    [
        {
            "split": "TRAIN",
            "mean_amount": train.TransactionAmt.mean(),
            **{f"p{int(q * 100)}_amount": value for q, value in train_amount_stats.items()},
        },
        {
            "split": "VALIDATION",
            "mean_amount": validation_amount.mean(),
            **{f"p{int(q * 100)}_amount": value for q, value in validation_amount_stats.items()},
        },
        {
            "split": "HOLDOUT",
            "mean_amount": artifact.TransactionAmt.mean(),
            **{f"p{int(q * 100)}_amount": value for q, value in holdout_amount_stats.items()},
        },
    ]
)
risk_edges = [0, 0.01, 0.02, 0.05, 0.10, 0.20, 0.40, 1.0000001]
risk_counts = np.histogram(artifact.calibrated_probability, bins=risk_edges)[0]
risk_table = pd.DataFrame(
    {
        "calibrated_risk_band": ["0-1%", "1-2%", "2-5%", "5-10%", "10-20%", "20-40%", "40%+"],
        "holdout_count": risk_counts,
        "holdout_share": risk_counts / len(artifact),
    }
)
validation_risk = pd.read_csv(ROOT / "reports/calibrated_risk_band_diagnostics.csv")
if len(validation_risk) == len(risk_table) and "count" in validation_risk:
    risk_table["validation_share"] = validation_risk["count"].to_numpy() / 88581
rule_text = ", ".join(
    f"{row.rule_id}={row.trigger_count:,} ({row.trigger_rate:.3%})"
    for row in rule_table.itertuples(index=False)
)
report = f"""# Final untouched HOLDOUT evaluation

## Pre-HOLDOUT lock verification

The committed working tree was clean at the start of this phase. The manifest was amended **before any HOLDOUT access** to explicitly lock the final model fit scope to TRAIN only. Its 467-feature pipeline, 0.8/0.2 blend, zero-day target-encoding label delay, frozen TRAIN-OOF-derived Platt calibrator, Balanced business costs, and rules-disabled primary policy were verified. The frozen rule-config SHA-256 is `3a4380831e177828c82a1a118259f83e354221fa52205b41d38aa21b58febb94`; the Platt artifact SHA-256 is `3503bf66161b5152e9de4d638b9e7932d581fc371abd6a45b6969ba6c8134f6e`. No HOLDOUT-derived artifact existed before this evaluation.

## Exact fit scope and chronological integrity

LightGBM, XGBoost, preprocessing, and label-based feature state were fitted on **TRAIN only**, using the locked parameters and temporal semantics. No TRAIN+VALIDATION refit occurred. The existing Platt artifact was reused unchanged. The HOLDOUT contains {len(artifact):,} rows, matching the original 88,581-row split. TransactionID is unique and disjoint from all TRAIN and VALIDATION IDs; its IDs and TransactionDT follow the development period. The feature schema matches TRAIN after excluding its label. No HOLDOUT label was available to feature construction, model scoring, calibration, business decisioning, or optional rule evaluation. Non-label history updates within the evaluation frame follow the existing feature transform implementation; label-based target encoding uses only matured TRAIN history.

## Prediction lock before label evaluation

The label-free `reports/holdout_predictions_frozen.csv` was written first with TransactionID, TransactionDT, TransactionAmt, both component scores, raw ensemble score, calibrated probability, and primary Balanced decision. It also records separately the fixed 5% capacity action and optional frozen-rule action. SHA-256: `{prediction_hash}`. `reports/holdout_prediction_lock.json` records the hash, row count, fit scope, and rules-disabled primary policy. Only after verifying this saved file and its checksum were HOLDOUT labels read and aligned by TransactionID.

## Final ranking and calibration

Primary PR-AUC {ranking["pr_auc"]:.6f}; ROC-AUC {ranking["roc_auc"]:.6f}. Versus frozen VALIDATION PR-AUC 0.615265 and ROC-AUC 0.916224, the absolute changes are {rank_table.pr_auc_absolute_change.iloc[0]:+.6f} and {rank_table.roc_auc_absolute_change.iloc[0]:+.6f}; relative changes are {rank_table.pr_auc_relative_change.iloc[0]:+.2%} and {rank_table.roc_auc_relative_change.iloc[0]:+.2%}. PR-AUC is the primary ranking metric.

Frozen Platt calibration: Brier {calibration["brier"]:.6f}; Log Loss {calibration["log_loss"]:.6f}; ten equal-count-bin ECE {calibration["ece"]:.6f}; mean probability {calibration["mean_probability"]:.6f}; observed fraud prevalence {calibration["prevalence"]:.6f}. Descriptive VALIDATION references: Brier 0.019493, Log Loss 0.082039, ECE 0.005434. No recalibration occurred. The reliability figure is `reports/figures/holdout_calibration_curve.png`.

At calibrated probability >= 0.50: {diagnostic_text}. **0.50 is a diagnostic cutoff, not the production business policy.**

## PRIMARY: Balanced expected-cost policy, rules disabled

{markdown_table(policy_table.iloc[:1], policy_columns)}

APPROVE/REVIEW/BLOCK counts and rates, modeled fraud transaction and amount capture, approved fraud count and amount, legitimate action rates, review workload, blocked count, illustrative total and per-transaction cost, approve-all cost, and modeled reduction versus approve-all are in the table. Fraud capture assumes the frozen illustrative REVIEW detection rate of 0.8; no actual review outcomes are observed. Modeled cost differences are not real-world savings.

### Fixed 5% review-capacity sensitivity

{markdown_table(policy_table.iloc[1:2], policy_columns)}

The uncapped primary and the already-established 5% cap use the same Balanced expected-cost formulas. REVIEW candidates are ranked by economic benefit and TransactionID; overflow receives the cheaper non-review action. No capacity was chosen from HOLDOUT outcomes.

## SECONDARY: OPTIONAL FROZEN OPERATIONAL OVERLAY

The rules were loaded unchanged from `reports/rules_frozen_config.json` and evaluated once. They are **not** part of the primary champion policy. Total triggered {rule_trigger_count:,}/{len(artifact):,} ({rule_trigger_count / len(artifact):.3%}); base-action overrides {rule_override_count:,}. Per-rule triggers: {rule_text}.

{markdown_table(policy_table.iloc[2:3], policy_columns)}

The overlay did not change calibrated probabilities. Its retrospective outcomes were not used to revise the rules or primary policy.

## VALIDATION-to-HOLDOUT generalization and distribution checks

{markdown_table(comparison, comparison_columns)}

Transaction-amount distribution (existing quantity, no new feature):

{markdown_table(amount_shift, list(amount_shift.columns))}

Calibrated-risk distribution compared with the earlier validation diagnostic bands:

{markdown_table(risk_table, list(risk_table.columns))}

These amount, risk, prevalence, and decision shifts are descriptive only. No drift-triggered system change was made.

## TRAIN-defined amount-band fraud outcomes

Bands use TRAIN amount quantiles only: {amount_edges.tolist()}. Captured fraud count and amount include BLOCK plus 0.8 times REVIEW, following the frozen illustrative assumption.

{markdown_table(band_table, list(band_table.columns))}

## Runtime, memory, and tracking

Runtime {runtime_seconds:.1f} seconds. Prepared feature-matrix estimate {matrix_mib:.1f} MiB. Kernel resident-memory snapshot at report time {kernel_rss_mib:.1f} MiB (not a peak). MLflow experiment `fraud-detection-final-holdout`; run `{run_id}`; tracking URI `{tracking_uri}`. The run logs frozen identifiers, terminal metrics, policy comparisons, capacity sensitivity, optional-rule results, runtime, and figures.

## Terminal result lock

**The HOLDOUT result is terminal.** No change to the model, features, ensemble, calibration, rules, thresholds, or business policy may be justified using HOLDOUT performance. Future improvements require a new experimental cycle and a new future evaluation set. No HOLDOUT-driven tuning occurred in this phase.
"""
(ROOT / "reports/final_holdout_evaluation_summary.md").write_text(report, encoding="utf-8")
print("Final report written")

Final report written
